In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import requests
import zipfile
import io
import plotly.express as px

In [1]:
INSTRUMENT_MAP = {
    # S&P 500 — all variants map to same instrument
    "ADJUSTED INT RATE S&P 500 TOTL - CHICAGO MERCANTILE EXCHANGE" :    "SP500",
    "E-MINI S&P 500 - CHICAGO MERCANTILE EXCHANGE" :                    "SP500",
    "E-MINI S&P 500 STOCK INDEX - CHICAGO MERCANTILE EXCHANGE" :        "SP500",
    "MICRO E-MINI S&P 500 INDEX - CHICAGO MERCANTILE EXCHANGE" :        "SP500",
    "S&P 500 ANNUAL DIVIDEND INDEX - CHICAGO MERCANTILE EXCHANGE" :     "SP500",
    "S&P 500 Consolidated - CHICAGO MERCANTILE EXCHANGE" :              "SP500",
    "S&P 500 QUARTERLY DIVIDEND IND - CHICAGO MERCANTILE EXCHANGE" :    "SP500",
    "S&P 500 STOCK INDEX - CHICAGO MERCANTILE EXCHANGE" :               "SP500",
    "S&P 500 TOTAL RETURN INDEX - CHICAGO MERCANTILE EXCHANGE" :        "SP500",

    'BITCOIN - CHICAGO MERCANTILE EXCHANGE':                            'BTC',
    'BITCOIN CASH PERP STYLE - COINBASE DERIVATIVES, LLC':              'BTC',
    'MICRO BITCOIN - CHICAGO MERCANTILE EXCHANGE':                      'BTC',
    'NANO BITCOIN PERP STYLE - COINBASE DERIVATIVES, LLC':              'BTC',

    "ULTRA 10-YEAR U.S. T-NOTES - CHICAGO BOARD OF TRADE" :             "US10Y",
    "ULTRA UST 10Y - CHICAGO BOARD OF TRADE" :                          "US10Y",
    "UST 10Y NOTE - CHICAGO BOARD OF TRADE" :                           "US10Y",
    "10-YEAR U.S. TREASURY NOTES - CHICAGO BOARD OF TRADE" :            "US10Y",
    
    '3-MONTH SOFR - CHICAGO MERCANTILE EXCHANGE' :                      "3 Month US",
    'SOFR-3M - CHICAGO MERCANTILE EXCHANGE' :                           "3 Month US",


    'SOFR-1M - CHICAGO MERCANTILE EXCHANGE' :                           '1 Month US',
    '1-MONTH SOFR - CHICAGO MERCANTILE EXCHANGE' :                      '1 Month US',

    "E-MINI RUSSELL 2000 INDEX - CHICAGO MERCANTILE EXCHANGE":          "russel",
    "EMINI RUSSELL 1000 GROWTH - CHICAGO MERCANTILE EXCHANGE":          "russel",
    "EMINI RUSSELL 1000 VALUE INDEX - CHICAGO MERCANTILE EXCHANGE":     "russel",
    "MICRO E-MINI RUSSELL 2000 INDX - CHICAGO MERCANTILE EXCHANGE":     "russel",
    "RUSSELL 1000 VALUE INDEX MINI - ICE FUTURES U.S.":                 "russel",
    "RUSSELL 2000 ANNUAL DIVIDEND  - CHICAGO MERCANTILE EXCHANGE":      "russel",
    "RUSSELL 2000 MINI INDEX FUTURE - ICE FUTURES U.S.":                "russel",
    "RUSSELL E-MINI - CHICAGO MERCANTILE EXCHANGE":                     "russel",

    'BRITISH POUND - CHICAGO MERCANTILE EXCHANGE':                      'GBP',
    'BRITISH POUND STERLING - CHICAGO MERCANTILE EXCHANGE':             'GBP',
    'EURO FX/BRITISH POUND XRATE - CHICAGO MERCANTILE EXCHANGE':        'GBP',

    'JAPANESE YEN - CHICAGO MERCANTILE EXCHANGE':                       'YEN',

    'CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE':                    "CAD",
    'SWISS FRANC - CHICAGO MERCANTILE EXCHANGE' :                       "CHF",

    "MICRO E-MINI NASDAQ-100 INDEX - CHICAGO MERCANTILE EXCHANGE" :     "NASDAQ",
    "NASDAQ MINI - CHICAGO MERCANTILE EXCHANGE" :                       "NASDAQ",
    "NASDAQ-100 Consolidated - CHICAGO MERCANTILE EXCHANGE" :           "NASDAQ",
    "NASDAQ-100 STOCK INDEX (MINI) - CHICAGO MERCANTILE EXCHANGE" :     "NASDAQ",
}

In [3]:
TICKER_MAP = {
    'SP500':        '^GSPC',
    'BTC':          'BTC-USD',
    'US10Y':        'IEF',
    '3 Month US' :  'SHV',
    '1 Month US' :  'SHV',
    'russel':       'IWM',
    'GBP':          'GBPUSD=X',
    'YEN':          'JPY=X',
    'CAD' :         'CAD=X',
    'CHF' :         'CHF=X',
    'NASDAQ':       'QQQ'
}

In [4]:
def download_cot_data(years, report_type='combined'):
    """
    Downloads raw CFTC Traders in Financial Futures (TFF) data.

    Args:
        years       : list of int — e.g. [2020, 2021, 2022]
        report_type : 'futures' or 'combined' (futures + options)

    Returns:
        pd.DataFrame — raw unmodified TFF data
    """
    prefix_map = {
        'futures':  'fut_fin_xls',
        'combined': 'com_fin_xls',
    }
    assert report_type in prefix_map, f"report_type must be one of {list(prefix_map.keys())}"
    prefix = prefix_map[report_type]

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
        'Referer':    'https://www.cftc.gov/MarketReports/CommitmentsofTraders/index.htm',
    }

    dfs = []
    for year in years:
        url = f"https://www.cftc.gov/files/dea/history/{prefix}_{year}.zip"
        print(f"Fetching {report_type} COT {year} from {url}...")
        try:
            response = requests.get(url, headers=headers, timeout=30)
            response.raise_for_status()
            zf  = zipfile.ZipFile(io.BytesIO(response.content))
            xls = zf.open(zf.namelist()[0]).read()
            dfs.append(pd.read_excel(io.BytesIO(xls), engine='xlrd'))
            print(f"  ✓ {year} fetched successfully")
        except requests.HTTPError as e:
            print(f"  ✗ HTTP error for {year}: {e}")
        except zipfile.BadZipFile:
            print(f"  ✗ Bad zip file for {year}")
        except Exception as e:
            print(f"  ✗ Unexpected error for {year}: {e}")

    if not dfs:
        raise RuntimeError("No data fetched — check years and network connection")

    return pd.concat(dfs, ignore_index=True)

In [19]:
def find_dominant_contracts(df, instrument_name):
    """
    For a given instrument, shows which contract names exist,
    how many weeks each was dominant, and the date range of dominance.
    
    Use this BEFORE updating INSTRUMENT_MAP to understand what to include.
    
    Args:
        df              : raw COT DataFrame
        instrument_name : str — search term e.g. 'EURO FX', 'NASDAQ', 'BITCOIN'
        top_n           : int — how many contracts to show
    
    Returns:
        pd.DataFrame — contract names, dominance count, date ranges
    """
    # Find all contracts matching the search term and print the variant names
    df_filtered = df[df['Market_and_Exchange_Names'].str.contains(instrument_name)]
    df_filtered = df_filtered[['Market_and_Exchange_Names', 'Report_Date_as_MM_DD_YYYY', 'Open_Interest_All']]
    df_filtered_pivot = df_filtered.pivot_table(
    index='Report_Date_as_MM_DD_YYYY',
    columns='Market_and_Exchange_Names',
    values='Open_Interest_All',
    aggfunc='first' ) # in case of duplicates, take first

    
    variant_names = (df_filtered_pivot.columns.values.tolist())
    print(f"Variants: For {instrument_name}: ") 
    for i in variant_names:
        print(i)
    if len(variant_names) == 0:
        print('Probably Searched Wrong')

    df_filtered_pivot['Total'] = df_filtered_pivot.sum(axis=1, )

    df_filtered_pivot_proportions = df_filtered_pivot.div(df_filtered_pivot['Total'], axis=0) * 100
    df_filtered_pivot_proportions = df_filtered_pivot_proportions.drop(columns='Total')
    return df_filtered_pivot_proportions
    #return px.line(df_filtered_pivot_proportions)

In [6]:
def clean_financial_cot_data(df, instrument_map):
    ''' Dealer — financial institutions/dealers (often on the other side of client trades)
        Asset_Mgr — institutional investors like pension funds, mutual funds
        Lev_Money — leveraged money (hedge funds, CTAs)
        Other_Rept — other reportable traders
        NonRept — non-reportable (small traders below reporting thresholds)
        Traders - it's counting number of traders, knowing 47 dealers are long tells you less than knowing their total position size
        Concentration Report - Shows what % of open interest the top 4 and top 8 traders control'''
    
    # take only the columns we need
    df = df[['Market_and_Exchange_Names', 
    'Report_Date_as_MM_DD_YYYY', 
    'Open_Interest_All', 
    'Dealer_Positions_Long_All', 'Dealer_Positions_Short_All',
    'Asset_Mgr_Positions_Long_All','Asset_Mgr_Positions_Short_All',
    'Lev_Money_Positions_Long_All', 'Lev_Money_Positions_Short_All'
    ]]


    df['Report_Date_as_MM_DD_YYYY'] = pd.to_datetime(df['Report_Date_as_MM_DD_YYYY'])
    df = df.sort_values('Report_Date_as_MM_DD_YYYY')

    df['Instrument'] = df['Market_and_Exchange_Names'].map(instrument_map)
    df = df[df['Instrument'].notna()].copy()

    cols_to_sum = df.columns.values.tolist()
    cols_to_sum.remove('Market_and_Exchange_Names')
    cols_to_sum.remove('Report_Date_as_MM_DD_YYYY')
    cols_to_sum.remove('Instrument')
    df = (
    df.groupby(['Instrument', 'Report_Date_as_MM_DD_YYYY'])[cols_to_sum]
    .sum()
    .reset_index()
    )


    return df

In [7]:
def process_COT(df_cot, instrument, start_date, end_date):
    """
    Args:
        df_cot      : cleaned COT dataframe (output of clean_cot_data())
        instrument  : str — e.g. 'SP500'
        start_date  : str — e.g. '2020-01-01'
        end_date    : str — e.g. '2022-12-31'
    """

    df_clean = df_cot[
    (df_cot['Instrument'] == instrument) &
    (df_cot['Report_Date_as_MM_DD_YYYY'] >= start_date) &
    (df_cot['Report_Date_as_MM_DD_YYYY'] <= end_date)
    ].copy()

    df_clean["Dealer Net"] = df_clean['Dealer_Positions_Long_All'] - df_clean['Dealer_Positions_Short_All']
    df_clean["Asset Manager Net"] = df_clean['Asset_Mgr_Positions_Long_All'] - df_clean['Asset_Mgr_Positions_Short_All'] 
    df_clean["Levered Net"] = df_clean['Lev_Money_Positions_Long_All'] - df_clean['Lev_Money_Positions_Short_All']

    df_clean["Dealer Long Proportion"] = df_clean['Dealer_Positions_Long_All']/df_clean['Open_Interest_All']
    df_clean["Asset Manager Long Proportion"] = df_clean['Asset_Mgr_Positions_Long_All']/df_clean['Open_Interest_All']
    df_clean["Levered Long Proportion"] = df_clean['Lev_Money_Positions_Long_All']/df_clean['Open_Interest_All']

    df_clean["Dealer Short Proportion"] = df_clean['Dealer_Positions_Short_All']/df_clean['Open_Interest_All']
    df_clean["Asset Manager Short Proportion"] = df_clean['Asset_Mgr_Positions_Short_All']/df_clean['Open_Interest_All']
    df_clean["Levered Short Proportion"] = df_clean['Lev_Money_Positions_Short_All']/df_clean['Open_Interest_All']


    df_clean["Dealer Crowding"] = (df_clean['Dealer_Positions_Long_All']+df_clean['Dealer_Positions_Short_All'])/df_clean['Open_Interest_All']
    df_clean["Asset Manager Crowding"] = (df_clean['Asset_Mgr_Positions_Long_All']+df_clean['Asset_Mgr_Positions_Short_All'])/df_clean['Open_Interest_All']
    df_clean["Levered Manager Crowding"] = (df_clean['Lev_Money_Positions_Long_All']+df_clean['Lev_Money_Positions_Short_All'])/df_clean['Open_Interest_All']

    ticker = TICKER_MAP[instrument]
    spy = yf.download(tickers=ticker, start=start_date, end=end_date)['Close']

    spy_tuesday = spy[spy.index.dayofweek == 1]
    spy_tuesday.index.name = 'Report_Date_as_MM_DD_YYYY'
    spy_tuesday.name = 'Price_Close'  # rename the Series before merging

    df_clean_merged = df_clean.merge(spy_tuesday, on='Report_Date_as_MM_DD_YYYY', how='inner')
    #df_clean_merged = df_clean_merged.rename(columns={ticker: 'Price_Close'})
    df_clean_merged

    return df_clean_merged

In [8]:
def feature_engineering(cot_clean):
    cot_clean["Dealer Net"] = cot_clean['Dealer_Positions_Long_All'] - cot_clean['Dealer_Positions_Short_All']
    cot_clean["Asset Manager Net"] = cot_clean['Asset_Mgr_Positions_Long_All'] - cot_clean['Asset_Mgr_Positions_Short_All'] 
    cot_clean["Levered Net"] = cot_clean['Lev_Money_Positions_Long_All'] - cot_clean['Lev_Money_Positions_Short_All']


    cot_clean["Dealer Ratio"] = cot_clean['Dealer_Positions_Long_All']/cot_clean['Dealer_Positions_Short_All']
    cot_clean["Asset Manager Ratio"] = cot_clean['Asset_Mgr_Positions_Long_All']/cot_clean['Asset_Mgr_Positions_Short_All'] 
    cot_clean["Levered Ratio"] = cot_clean['Lev_Money_Positions_Long_All']/cot_clean['Lev_Money_Positions_Short_All']


    cot_clean["Dealer Long Proportion"] = cot_clean['Dealer_Positions_Long_All']/cot_clean['Open_Interest_All']
    cot_clean["Asset Manager Long Proportion"] = cot_clean['Asset_Mgr_Positions_Long_All']/cot_clean['Open_Interest_All']
    cot_clean["Levered Long Proportion"] = cot_clean['Lev_Money_Positions_Long_All']/cot_clean['Open_Interest_All']

    cot_clean["Dealer Short Proportion"] = cot_clean['Dealer_Positions_Short_All']/cot_clean['Open_Interest_All']
    cot_clean["Asset Manager Short Proportion"] = cot_clean['Asset_Mgr_Positions_Short_All']/cot_clean['Open_Interest_All']
    cot_clean["Levered Short Proportion"] = cot_clean['Lev_Money_Positions_Short_All']/cot_clean['Open_Interest_All']


    cot_clean["Dealer Crowding"] = (cot_clean['Dealer_Positions_Long_All']+cot_clean['Dealer_Positions_Short_All'])/cot_clean['Open_Interest_All']
    cot_clean["Asset Manager Crowding"] = (cot_clean['Asset_Mgr_Positions_Long_All']+cot_clean['Asset_Mgr_Positions_Short_All'])/cot_clean['Open_Interest_All']
    cot_clean["Levered Manager Crowding"] = (cot_clean['Lev_Money_Positions_Long_All']+cot_clean['Lev_Money_Positions_Short_All'])/cot_clean['Open_Interest_All']
    return cot_clean

In [9]:
def view_dates(cot_clean):
    start_dates = []
    end_dates = []
    for instrument in cot_clean['Instrument'].unique().tolist():
        temp = cot_clean[cot_clean['Instrument'] == instrument]
        print(f'{instrument}: ({temp.iloc[0,1]}  <->  {temp.iloc[-1,1]})')
        start_dates.append(temp.iloc[0,1])
        end_dates.append(temp.iloc[-1,1])
    return start_dates, end_dates

In [10]:
def cot_treemap(variable, tf_in_weeks, cot_clean, prices):
    
    df_corr = pd.DataFrame()

    for instrument in cot_clean['Instrument'].unique().tolist():
        ### this is too long but it works
        temp = pd.DataFrame(cot_clean[cot_clean['Instrument'] == instrument].set_index('Report_Date_as_MM_DD_YYYY').drop(columns = 'Instrument')[variable]).rename(columns={variable : instrument})
        df_corr = df_corr.join(temp, how="outer")

    df_corr.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_corr = df_corr.dropna()

    df_corr

    prices_clean = prices.ffill()
    prices_clean = prices_clean.dropna()
    prices_clean = prices_clean.reindex(df_corr.index).dropna()

    returns_dict = dict(prices_clean.pct_change(periods=tf_in_weeks).iloc[-1]*100)

    df_corr_treemap = pd.DataFrame(index = df_corr.columns.values.tolist())
    df_corr_treemap[variable] = df_corr.iloc[-1]
    df_corr_treemap = df_corr_treemap.reset_index()
    df_corr_treemap['parent'] = ''
    df_corr_treemap['ticker'] = df_corr_treemap['index'].map(TICKER_MAP)
    df_corr_treemap['return'] = df_corr_treemap['ticker'].map(returns_dict)
    df_corr_treemap['absolute value'] = abs(df_corr_treemap[variable])
    return df_corr_treemap

In [11]:
ticker_list = []
for tick in TICKER_MAP:
    ticker_list.append((TICKER_MAP[tick]))

ticker_list

['^GSPC',
 'BTC-USD',
 'IEF',
 'SHV',
 'SHV',
 'IWM',
 'GBPUSD=X',
 'JPY=X',
 'CAD=X',
 'CHF=X',
 'QQQ']

In [12]:
ohlc = yf.download(tickers=ticker_list, start = '2018-01-01')
ohlc

/var/folders/hs/v5c0lrnj61z3tp745vdhh0m40000gn/T/ipykernel_48363/244945367.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  ohlc = yf.download(tickers=ticker_list, start = '2018-01-01')
[*********************100%***********************]  10 of 10 completed


Price              Close                                                     \
Ticker           BTC-USD    CAD=X    CHF=X  GBPUSD=X        IEF         IWM   
Date                                                                          
2018-01-01  13657.200195  1.25808  0.97373  1.351607        NaN         NaN   
2018-01-02  14982.099609  1.25550  0.97472  1.351132  86.685112  138.884445   
2018-01-03  15201.000000  1.25039  0.97146  1.359619  86.775742  139.028793   
2018-01-04  15599.200195  1.25469  0.97714  1.351132  86.734581  139.398788   
2018-01-05  17429.500000  1.24903  0.97440  1.355289  86.627441  139.687607   
...                  ...      ...      ...       ...        ...         ...   
2026-04-12  70753.406250      NaN      NaN       NaN        NaN         NaN   
2026-04-13  74484.640625  1.38727  0.79261  1.339441  95.480003  265.070007   
2026-04-14  74181.609375  1.37850  0.78320  1.351534  95.790001  268.720001   
2026-04-15  74789.250000  1.37661  0.78091  1.357534  95.580002  269.390015   
2026-04-16           NaN  1.37336  0.78133  1.357010        NaN         NaN   

Price                                                        ...  \
Ticker           JPY=X         QQQ         SHV        ^GSPC  ...   
Date                                                         ...   
2018-01-01  112.666000         NaN         NaN          NaN  ...   
2018-01-02  112.769997  150.222168   89.637917  2695.810059  ...   
2018-01-03  112.244003  151.681808   89.637917  2713.060059  ...   
2018-01-04  112.607002  151.947174   89.654167  2723.989990  ...   
2018-01-05  112.782997  153.473221   89.654167  2743.149902  ...   
...                ...         ...         ...          ...  ...   
2026-04-12         NaN         NaN         NaN          NaN  ...   
2026-04-13  159.679993  617.390015  110.199997  6886.240234  ...   
2026-04-14  159.214005  628.599976  110.220001  6967.379883  ...   
2026-04-15  158.792007  637.400024  110.220001  7022.950195  ...   
2026-04-16  158.787003         NaN         NaN          NaN  ...   

Price             Volume                                                    \
Ticker           BTC-USD CAD=X CHF=X GBPUSD=X        IEF         IWM JPY=X   
Date                                                                         
2018-01-01  1.029120e+10   0.0   0.0      0.0        NaN         NaN   0.0   
2018-01-02  1.684660e+10   0.0   0.0      0.0  3902000.0  20489600.0   0.0   
2018-01-03  1.687190e+10   0.0   0.0      0.0  2135900.0  21836600.0   0.0   
2018-01-04  2.178320e+10   0.0   0.0      0.0  2430000.0  14207100.0   0.0   
2018-01-05  2.384090e+10   0.0   0.0      0.0  2163600.0  19883900.0   0.0   
...                  ...   ...   ...      ...        ...         ...   ...   
2026-04-12  2.988274e+10   NaN   NaN      NaN        NaN         NaN   NaN   
2026-04-13  5.227821e+10   0.0   0.0      0.0  4300200.0  23820000.0   0.0   
2026-04-14  5.354083e+10   0.0   0.0      0.0  5239800.0  24270400.0   0.0   
2026-04-15  3.763494e+10   0.0   0.0      0.0  5592731.0  21590055.0   0.0   
2026-04-16           NaN   0.0   0.0      0.0        NaN         NaN   0.0   

Price                                            
Ticker             QQQ        SHV         ^GSPC  
Date                                             
2018-01-01         NaN        NaN           NaN  
2018-01-02  32573300.0   692300.0  3.397430e+09  
2018-01-03  29383600.0   546700.0  3.544030e+09  
2018-01-04  24776100.0   553000.0  3.697340e+09  
2018-01-05  26992300.0   240100.0  3.239280e+09  
...                ...        ...           ...  
2026-04-12         NaN        NaN           NaN  
2026-04-13  32972100.0  3801300.0  4.785840e+09  
2026-04-14  49921100.0  3486200.0  5.032380e+09  
2026-04-15  46965650.0  3310431.0  3.062289e+09  
2026-04-16         NaN        NaN           NaN  

[3028 rows x 50 columns]

In [14]:
prices = ohlc['Close']
prices

Ticker,BTC-USD,CAD=X,CHF=X,GBPUSD=X,IEF,IWM,JPY=X,QQQ,SHV,^GSPC
Date,,,,,,,,,,
2018-01-01,13657.200195,1.25808,0.97373,1.351607,NaN,NaN,112.666000,NaN,NaN,NaN
2018-01-02,14982.099609,1.25550,0.97472,1.351132,86.685112,138.884445,112.769997,150.222168,89.637917,2695.810059
2018-01-03,15201.000000,1.25039,0.97146,1.359619,86.775742,139.028793,112.244003,151.681808,89.637917,2713.060059
2018-01-04,15599.200195,1.25469,0.97714,1.351132,86.734581,139.398788,112.607002,151.947174,89.654167,2723.989990
2018-01-05,17429.500000,1.24903,0.97440,1.355289,86.627441,139.687607,112.782997,153.473221,89.654167,2743.149902
...,...,...,...,...,...,...,...,...,...,...
2026-04-12,70753.406250,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-04-13,74484.640625,1.38727,0.79261,1.339441,95.480003,265.070007,159.679993,617.390015,110.199997,6886.240234
2026-04-14,74181.609375,1.37850,0.78320,1.351534,95.790001,268.720001,159.214005,628.599976,110.220001,6967.379883


In [15]:
volume = ohlc['Volume']
volume

Ticker,BTC-USD,CAD=X,CHF=X,GBPUSD=X,IEF,IWM,JPY=X,QQQ,SHV,^GSPC
Date,,,,,,,,,,
2018-01-01,1.029120e+10,0.0,0.0,0.0,NaN,NaN,0.0,NaN,NaN,NaN
2018-01-02,1.684660e+10,0.0,0.0,0.0,3902000.0,20489600.0,0.0,32573300.0,692300.0,3.397430e+09
2018-01-03,1.687190e+10,0.0,0.0,0.0,2135900.0,21836600.0,0.0,29383600.0,546700.0,3.544030e+09
2018-01-04,2.178320e+10,0.0,0.0,0.0,2430000.0,14207100.0,0.0,24776100.0,553000.0,3.697340e+09
2018-01-05,2.384090e+10,0.0,0.0,0.0,2163600.0,19883900.0,0.0,26992300.0,240100.0,3.239280e+09
...,...,...,...,...,...,...,...,...,...,...
2026-04-12,2.988274e+10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-04-13,5.227821e+10,0.0,0.0,0.0,4300200.0,23820000.0,0.0,32972100.0,3801300.0,4.785840e+09
2026-04-14,5.354083e+10,0.0,0.0,0.0,5239800.0,24270400.0,0.0,49921100.0,3486200.0,5.032380e+09


In [17]:
cot_raw = download_cot_data([2017,2018,2019,2020, 2021, 2022, 2023, 2024, 2025, 2026])
cot_raw

Fetching combined COT 2017 from https://www.cftc.gov/files/dea/history/com_fin_xls_2017.zip...
  ✓ 2017 fetched successfully
Fetching combined COT 2018 from https://www.cftc.gov/files/dea/history/com_fin_xls_2018.zip...
  ✓ 2018 fetched successfully
Fetching combined COT 2019 from https://www.cftc.gov/files/dea/history/com_fin_xls_2019.zip...
  ✓ 2019 fetched successfully
Fetching combined COT 2020 from https://www.cftc.gov/files/dea/history/com_fin_xls_2020.zip...
  ✓ 2020 fetched successfully
Fetching combined COT 2021 from https://www.cftc.gov/files/dea/history/com_fin_xls_2021.zip...
  ✓ 2021 fetched successfully
Fetching combined COT 2022 from https://www.cftc.gov/files/dea/history/com_fin_xls_2022.zip...
  ✓ 2022 fetched successfully
Fetching combined COT 2023 from https://www.cftc.gov/files/dea/history/com_fin_xls_2023.zip...
  ✓ 2023 fetched successfully
Fetching combined COT 2024 from https://www.cftc.gov/files/dea/history/com_fin_xls_2024.zip...
  ✓ 2024 fetched successfully


,Market_and_Exchange_Names,As_of_Date_In_Form_YYMMDD,Report_Date_as_MM_DD_YYYY,CFTC_Contract_Market_Code,CFTC_Market_Code,CFTC_Region_Code,CFTC_Commodity_Code,Open_Interest_All,Dealer_Positions_Long_All,Dealer_Positions_Short_All,...,Conc_Gross_LE_4_TDR_Short_All,Conc_Gross_LE_8_TDR_Long_All,Conc_Gross_LE_8_TDR_Short_All,Conc_Net_LE_4_TDR_Long_All,Conc_Net_LE_4_TDR_Short_All,Conc_Net_LE_8_TDR_Long_All,Conc_Net_LE_8_TDR_Short_All,Contract_Units,CFTC_SubGroup_Code,FutOnly_or_Combined
0,CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE,171226,2017-12-26,090741,CME,0,90,134238,4812,55036,...,41.9,33.7,51.5,20.4,41.3,30.7,50.4,"(CONTRACTS OF CAD 100,000)",F10,Combined
1,CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE,171219,2017-12-19,090741,CME,0,90,172676,15819,81942,...,46.3,35.2,56.2,20.0,42.0,32.3,51.4,"(CONTRACTS OF CAD 100,000)",F10,Combined
2,CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE,171212,2017-12-12,090741,CME,0,90,165499,16251,79621,...,45.0,32.6,55.8,19.1,43.9,30.7,53.4,"(CONTRACTS OF CAD 100,000)",F10,Combined
3,CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE,171205,2017-12-05,090741,CME,0,90,171861,12245,82228,...,44.3,31.7,55.9,18.3,43.9,29.5,53.9,"(CONTRACTS OF CAD 100,000)",F10,Combined
4,CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE,171128,2017-11-28,090741,CME,0,90,170364,7049,84530,...,45.1,31.7,56.5,20.9,44.9,28.7,54.2,"(CONTRACTS OF CAD 100,000)",F10,Combined
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25692,BBG COMMODITY - CHICAGO BOARD OF TRADE,260127,2026-01-27,221602,CBT,0,221,188271,125275,172650,...,89.4,89.8,96.8,82.8,89.4,87.9,95.4,($100 X INDEX),F90,Combined
25693,BBG COMMODITY - CHICAGO BOARD OF TRADE,260120,2026-01-20,221602,CBT,0,221,189350,125416,171958,...,88.5,89.7,96.0,82.6,88.5,87.7,94.8,($100 X INDEX),F90,Combined
25694,BBG COMMODITY - CHICAGO BOARD OF TRADE,260113,2026-01-13,221602,CBT,0,221,189092,125364,171894,...,88.6,89.7,96.2,82.7,88.6,87.8,95.0,($100 X INDEX),F90,Combined
25695,BBG COMMODITY - CHICAGO BOARD OF TRADE,260106,2026-01-06,221602,CBT,0,221,204353,124995,187085,...,90.5,88.1,96.5,77.6,90.5,86.5,95.0,($100 X INDEX),F90,Combined


In [20]:
dominant_contract_analysis = find_dominant_contracts(cot_raw, 'NAS')
px.line(dominant_contract_analysis)

Variants: For NAS: 
MICRO E-MINI NASDAQ-100 INDEX - CHICAGO MERCANTILE EXCHANGE
NASDAQ MINI - CHICAGO MERCANTILE EXCHANGE
NASDAQ-100 Consolidated - CHICAGO MERCANTILE EXCHANGE
NASDAQ-100 STOCK INDEX (MINI) - CHICAGO MERCANTILE EXCHANGE


In [21]:
cot_clean = clean_financial_cot_data(cot_raw, INSTRUMENT_MAP)
cot_clean

/var/folders/hs/v5c0lrnj61z3tp745vdhh0m40000gn/T/ipykernel_48363/2476550818.py:20: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,Instrument,Report_Date_as_MM_DD_YYYY,Open_Interest_All,Dealer_Positions_Long_All,Dealer_Positions_Short_All,Asset_Mgr_Positions_Long_All,Asset_Mgr_Positions_Short_All,Lev_Money_Positions_Long_All,Lev_Money_Positions_Short_All
0,1 Month US,2018-07-03,5921,3312,609,0,0,1,2794
1,1 Month US,2018-07-10,6670,3481,683,0,0,35,2693
2,1 Month US,2018-07-17,8879,5557,2356,0,0,28,3233
3,1 Month US,2018-07-24,12655,7410,3562,0,0,748,4773
4,1 Month US,2018-07-31,14365,9576,4733,0,0,242,5186
...,...,...,...,...,...,...,...,...,...
5093,russel,2026-03-10,497841,116253,105045,179122,134544,91387,132519
5094,russel,2026-03-17,586922,144519,109868,178574,134742,81044,139963
5095,russel,2026-03-24,478081,122590,95247,166950,135445,88922,142696
5096,russel,2026-03-31,472964,140220,103142,160678,148025,74459,116169


In [22]:
cot_clean = feature_engineering(cot_clean)
cot_clean

,Instrument,Report_Date_as_MM_DD_YYYY,Open_Interest_All,Dealer_Positions_Long_All,Dealer_Positions_Short_All,Asset_Mgr_Positions_Long_All,Asset_Mgr_Positions_Short_All,Lev_Money_Positions_Long_All,Lev_Money_Positions_Short_All,Dealer Net,...,Levered Ratio,Dealer Long Proportion,Asset Manager Long Proportion,Levered Long Proportion,Dealer Short Proportion,Asset Manager Short Proportion,Levered Short Proportion,Dealer Crowding,Asset Manager Crowding,Levered Manager Crowding
0,1 Month US,2018-07-03,5921,3312,609,0,0,1,2794,2703,...,0.000358,0.559365,0.000000,0.000169,0.102854,0.000000,0.471880,0.662219,0.000000,0.472049
1,1 Month US,2018-07-10,6670,3481,683,0,0,35,2693,2798,...,0.012997,0.521889,0.000000,0.005247,0.102399,0.000000,0.403748,0.624288,0.000000,0.408996
2,1 Month US,2018-07-17,8879,5557,2356,0,0,28,3233,3201,...,0.008661,0.625859,0.000000,0.003154,0.265345,0.000000,0.364118,0.891204,0.000000,0.367271
3,1 Month US,2018-07-24,12655,7410,3562,0,0,748,4773,3848,...,0.156715,0.585539,0.000000,0.059107,0.281470,0.000000,0.377163,0.867009,0.000000,0.436270
4,1 Month US,2018-07-31,14365,9576,4733,0,0,242,5186,4843,...,0.046664,0.666620,0.000000,0.016847,0.329481,0.000000,0.361016,0.996102,0.000000,0.377863
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5093,russel,2026-03-10,497841,116253,105045,179122,134544,91387,132519,11208,...,0.689614,0.233514,0.359798,0.183567,0.211001,0.270255,0.266187,0.444515,0.630053,0.449754
5094,russel,2026-03-17,586922,144519,109868,178574,134742,81044,139963,34651,...,0.579039,0.246232,0.304255,0.138083,0.187194,0.229574,0.238470,0.433426,0.533829,0.376553
5095,russel,2026-03-24,478081,122590,95247,166950,135445,88922,142696,27343,...,0.623157,0.256421,0.349209,0.185998,0.199228,0.283310,0.298477,0.455649,0.632518,0.484474
5096,russel,2026-03-31,472964,140220,103142,160678,148025,74459,116169,37078,...,0.640954,0.296471,0.339726,0.157431,0.218076,0.312973,0.245619,0.514547,0.652699,0.403050


In [29]:
cot_clean['Instrument'].unique().tolist()

['1 Month US',
 '3 Month US',
 'BTC',
 'CAD',
 'CHF',
 'GBP',
 'NASDAQ',
 'SP500',
 'US10Y',
 'YEN',
 'russel']

 + 'Open_Interest_All': Open Positions
 + 'Dealer_Positions_Long_All' : Dealer Longs
 + 'Dealer_Positions_Short_All' : Dealer Shorts
 + 'Asset_Mgr_Positions_Long_All' : Asset Manager Longs
 + 'Asset_Mgr_Positions_Short_All' : Asset Manager Shorts
 + 'Lev_Money_Positions_Long_All' : Hedge Fund Longs
 + 'Lev_Money_Positions_Short_All' : Hedge Fund Shorts
 + 'Dealer Net' : Dealer Longs - Dealer Shorts
 + 'Asset Manager Net' : Asset Manager Longs - Asset Manager Shorts
 + 'Levered Net' : Hedge Fund Longs - Hedge Fund Shorts
 + 'Dealer Ratio' : Dealer Longs/ Dealer Shorts
 + 'Asset Manager Ratio' : Asset Manager Longs/ Asset Manager Shorts
 + 'Levered Ratio': Hedge Fund Longs/ Hedge Fund Shorts
 + 'Dealer Long Proportion' : Dealer Longs/Dealer Positions
 + 'Asset Manager Long Proportion' : Asset Manager Longs/Asset Manager Positions
 + 'Levered Long Proportion' : Levered Longs/Levered Positions
 + 'Dealer Short Proportion' : Dealer Short/Dealer Positions
 + 'Asset Manager Short Proportion' : Asset Manager Short/Asset Manager Positions
 + 'Levered Short Proportion' : Levered/Asset Manager Positions
 + 'Dealer Crowding' : Dealer Longs + Dealer Short / all positions
 + 'Asset Manager Crowding' : Asset Manager Longs + Asset Manager Short / all positions
 + 'Levered Crowding' : Levered Longs + Levered Short / all positions

In [31]:
cot_clean.columns.values.tolist()

['Instrument',
 'Report_Date_as_MM_DD_YYYY',
 'Open_Interest_All',
 'Dealer_Positions_Long_All',
 'Dealer_Positions_Short_All',
 'Asset_Mgr_Positions_Long_All',
 'Asset_Mgr_Positions_Short_All',
 'Lev_Money_Positions_Long_All',
 'Lev_Money_Positions_Short_All',
 'Dealer Net',
 'Asset Manager Net',
 'Levered Net',
 'Dealer Ratio',
 'Asset Manager Ratio',
 'Levered Ratio',
 'Dealer Long Proportion',
 'Asset Manager Long Proportion',
 'Levered Long Proportion',
 'Dealer Short Proportion',
 'Asset Manager Short Proportion',
 'Levered Short Proportion',
 'Dealer Crowding',
 'Asset Manager Crowding',
 'Levered Manager Crowding']

In [30]:
cot_clean[cot_clean['Instrument'] == '1 Month US']

,Instrument,Report_Date_as_MM_DD_YYYY,Open_Interest_All,Dealer_Positions_Long_All,Dealer_Positions_Short_All,Asset_Mgr_Positions_Long_All,Asset_Mgr_Positions_Short_All,Lev_Money_Positions_Long_All,Lev_Money_Positions_Short_All,Dealer Net,...,Levered Ratio,Dealer Long Proportion,Asset Manager Long Proportion,Levered Long Proportion,Dealer Short Proportion,Asset Manager Short Proportion,Levered Short Proportion,Dealer Crowding,Asset Manager Crowding,Levered Manager Crowding
0,1 Month US,2018-07-03,5921,3312,609,0,0,1,2794,2703,...,0.000358,0.559365,0.000000,0.000169,0.102854,0.000000,0.471880,0.662219,0.000000,0.472049
1,1 Month US,2018-07-10,6670,3481,683,0,0,35,2693,2798,...,0.012997,0.521889,0.000000,0.005247,0.102399,0.000000,0.403748,0.624288,0.000000,0.408996
2,1 Month US,2018-07-17,8879,5557,2356,0,0,28,3233,3201,...,0.008661,0.625859,0.000000,0.003154,0.265345,0.000000,0.364118,0.891204,0.000000,0.367271
3,1 Month US,2018-07-24,12655,7410,3562,0,0,748,4773,3848,...,0.156715,0.585539,0.000000,0.059107,0.281470,0.000000,0.377163,0.867009,0.000000,0.436270
4,1 Month US,2018-07-31,14365,9576,4733,0,0,242,5186,4843,...,0.046664,0.666620,0.000000,0.016847,0.329481,0.000000,0.361016,0.996102,0.000000,0.377863
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
401,1 Month US,2026-03-10,1368395,514066,179921,78096,32028,172841,555903,334145,...,0.310919,0.375671,0.057071,0.126309,0.131483,0.023406,0.406245,0.507154,0.080477,0.532554
402,1 Month US,2026-03-17,1467622,540939,198520,89321,31756,188023,595342,342419,...,0.315824,0.368582,0.060861,0.128114,0.135266,0.021638,0.405651,0.503848,0.082499,0.533765
403,1 Month US,2026-03-24,1546170,543509,239959,151861,30709,141078,574505,303550,...,0.245564,0.351520,0.098218,0.091244,0.155196,0.019861,0.371567,0.506715,0.118079,0.462810
404,1 Month US,2026-03-31,1645019,521528,336275,127221,31450,201586,491435,185253,...,0.410199,0.317035,0.077337,0.122543,0.204420,0.019118,0.298741,0.521455,0.096455,0.421284


In [40]:
def ordinal(n):
    n = int(n)
    if 11 <= n % 100 <= 13:
        return f"{n}th"
    return f"{n}{['th','st','nd','rd'][min(n % 10, 4) if n % 10 < 4 else 0]}"


def percentile_rank(series, value):
    clean = series.replace([np.inf, -np.inf], np.nan).dropna()
    if len(clean) == 0:
        return 50.0
    return (clean < value).mean() * 100


GROUPS = [
    {
        'name': 'Dealer',
        'net': 'Dealer Net',
        'ratio': 'Dealer Ratio',
        'crowding': 'Dealer Crowding',
        'long_prop': 'Dealer Long Proportion',
        'short_prop': 'Dealer Short Proportion',
        'long_abs': 'Dealer_Positions_Long_All',
        'short_abs': 'Dealer_Positions_Short_All',
    },
    {
        'name': 'Asset Manager',
        'net': 'Asset Manager Net',
        'ratio': 'Asset Manager Ratio',
        'crowding': 'Asset Manager Crowding',
        'long_prop': 'Asset Manager Long Proportion',
        'short_prop': 'Asset Manager Short Proportion',
        'long_abs': 'Asset_Mgr_Positions_Long_All',
        'short_abs': 'Asset_Mgr_Positions_Short_All',
    },
    {
        'name': 'Hedge Fund',
        'net': 'Levered Net',
        'ratio': 'Levered Ratio',
        'crowding': 'Levered Manager Crowding',
        'long_prop': 'Levered Long Proportion',
        'short_prop': 'Levered Short Proportion',
        'long_abs': 'Lev_Money_Positions_Long_All',
        'short_abs': 'Lev_Money_Positions_Short_All',
    },
]


def generate_insights(cot_clean, lookback_weeks=104):
    insights = []
    flagged = set()  # prevent duplicate signals
    latest = cot_clean.groupby('Instrument').last()

    for inst, row in latest.iterrows():
        hist = cot_clean[cot_clean['Instrument'] == inst].tail(lookback_weeks)
        has_prev = len(hist) >= 2
        prev = hist.iloc[-2] if has_prev else None

        for g in GROUPS:

            # ── 1. WEEKLY FLIP ────────────────────────────────────
            # Did this group cross from net long to net short (or vice versa)?
            # Confirmed by checking the sign of net position changed
            if has_prev and prev[g['net']] * row[g['net']] < 0:
                direction = 'net long' if row[g['net']] > 0 else 'net short'
                insights.append({
                    'type': 'flip',
                    'instrument': inst,
                    'text': f"{g['name']}s flipped {inst} to {direction} this week",
                    'severity': 'high',
                })

            # ── 2. DEALER NOT HEDGING ─────────────────────────────
            # Dealers normally intermediate client flow — when they step back
            # or break their typical posture, something structural changed
            if g['name'] == 'Dealer':
                crowding_pctl = percentile_rank(hist[g['crowding']], row[g['crowding']])
                net_pctl = percentile_rank(hist[g['net']], row[g['net']])

                # Dealers pulling back entirely
                if crowding_pctl < 10:
                    insights.append({
                        'type': 'dealer_absent',
                        'instrument': inst,
                        'text': f"Dealers pulling back on {inst} — crowding at {row[g['crowding']]:.0%} of OI ({ordinal(crowding_pctl)} percentile). Reduced hedging activity",
                        'severity': 'high',
                    })
                    flagged.add((inst, 'Dealer', 'crowding_low'))

                # Dealers on an unusual side of the trade
                if net_pctl > 95:
                    insights.append({
                        'type': 'dealer_unusual',
                        'instrument': inst,
                        'text': f"Dealers unusually net long on {inst} ({ordinal(net_pctl)} percentile) — not in typical hedging posture",
                        'severity': 'high',
                    })
                elif net_pctl < 5:
                    insights.append({
                        'type': 'dealer_unusual',
                        'instrument': inst,
                        'text': f"Dealers extremely net short on {inst} ({ordinal(net_pctl)} percentile) — heavy hedging or directional bet",
                        'severity': 'high',
                    })

            # ── 3. RATIO EXTREMES ─────────────────────────────────
            # Long/short ratio at historical extremes = one-sided bet
            # Skip inf/nan (happens when short side is zero)
            current_ratio = row[g['ratio']]
            if not (np.isinf(current_ratio) or np.isnan(current_ratio)):
                ratio_pctl = percentile_rank(hist[g['ratio']], current_ratio)

                if ratio_pctl > 95:
                    if current_ratio > 1:
                        label = "extreme bullish positioning"
                    else:
                        label = "least bearish on record — still net short"
                    insights.append({
                        'type': 'ratio_extreme',
                        'instrument': inst,
                        'text': f"{g['name']} long/short ratio on {inst} at {current_ratio:.2f}x — {ordinal(ratio_pctl)} percentile, {label}",
                        'severity': 'high',
                    })

                elif ratio_pctl < 5:
                    if current_ratio < 1:
                        label = "extreme bearish positioning"
                    else:
                        label = "least bullish on record — still net long"
                    insights.append({
                        'type': 'ratio_extreme',
                        'instrument': inst,
                        'text': f"{g['name']} long/short ratio on {inst} at {current_ratio:.2f}x — {ordinal(ratio_pctl)} percentile, {label}",
                        'severity': 'high',
                    })

            # ── 4. CROWDING EXTREMES ──────────────────────────────
            # One group dominating open interest at historical highs/lows
            # Skip if already flagged by dealer_absent check
            crowding_pctl = percentile_rank(hist[g['crowding']], row[g['crowding']])
            if crowding_pctl > 90:
                insights.append({
                    'type': 'crowding',
                    'instrument': inst,
                    'text': f"{g['name']} crowding on {inst} at {ordinal(crowding_pctl)} percentile ({row[g['crowding']]:.0%} of OI) — historically elevated",
                    'severity': 'high',
                })
            elif crowding_pctl < 10 and (inst, g['name'], 'crowding_low') not in flagged:
                insights.append({
                    'type': 'crowding',
                    'instrument': inst,
                    'text': f"{g['name']} crowding on {inst} at {ordinal(crowding_pctl)} percentile ({row[g['crowding']]:.0%} of OI) — unusually low participation",
                    'severity': 'medium',
                })

            # ── 5. PROPORTION DIVERGENCE ──────────────────────────
            # Two groups on opposite sides — meaningful only when proportions are large
            # Check dealer vs hedge fund, and asset manager vs hedge fund
            if g['name'] == 'Dealer':
                hf = GROUPS[2]  # Hedge Fund
                d_long = row[g['long_prop']]
                d_short = row[g['short_prop']]
                hf_long = row[hf['long_prop']]
                hf_short = row[hf['short_prop']]

                if d_long > 0.4 and hf_short > 0.3:
                    insights.append({
                        'type': 'divergence',
                        'instrument': inst,
                        'text': f"Dealers hold {d_long:.0%} of longs while Hedge Funds hold {hf_short:.0%} of shorts on {inst} — strong divergence",
                        'severity': 'high',
                    })
                elif d_short > 0.4 and hf_long > 0.3:
                    insights.append({
                        'type': 'divergence',
                        'instrument': inst,
                        'text': f"Dealers hold {d_short:.0%} of shorts while Hedge Funds hold {hf_long:.0%} of longs on {inst} — strong divergence",
                        'severity': 'high',
                    })

            if g['name'] == 'Asset Manager':
                hf = GROUPS[2]
                am_long = row[g['long_prop']]
                hf_short = row[hf['short_prop']]
                am_short = row[g['short_prop']]
                hf_long = row[hf['long_prop']]

                if am_long > 0.35 and hf_short > 0.25:
                    insights.append({
                        'type': 'divergence',
                        'instrument': inst,
                        'text': f"Asset Managers ({am_long:.0%} of longs) vs Hedge Funds ({hf_short:.0%} of shorts) on {inst}",
                        'severity': 'medium',
                    })
                elif am_short > 0.35 and hf_long > 0.25:
                    insights.append({
                        'type': 'divergence',
                        'instrument': inst,
                        'text': f"Asset Managers ({am_short:.0%} of shorts) vs Hedge Funds ({hf_long:.0%} of longs) on {inst}",
                        'severity': 'medium',
                    })

            # ── 6. WEEKLY PROPORTION SHIFTS ───────────────────────
            # Big moves in proportion, confirmed by absolute contract change
            # Both must agree — prevents false signals from OI shrinkage
            if has_prev:
                long_shift = row[g['long_prop']] - prev[g['long_prop']]
                long_delta = row[g['long_abs']] - prev[g['long_abs']]
                short_shift = row[g['short_prop']] - prev[g['short_prop']]
                short_delta = row[g['short_abs']] - prev[g['short_abs']]

                if abs(long_shift) > 0.05 and long_shift * long_delta > 0:
                    direction = 'added' if long_shift > 0 else 'cut'
                    insights.append({
                        'type': 'shift',
                        'instrument': inst,
                        'text': f"{g['name']}s {direction} longs on {inst} — proportion moved {long_shift:+.1%} of OI, backed by {abs(long_delta):,.0f} contracts",
                        'severity': 'medium',
                    })

                if abs(short_shift) > 0.05 and short_shift * short_delta > 0:
                    direction = 'added' if short_shift > 0 else 'cut'
                    insights.append({
                        'type': 'shift',
                        'instrument': inst,
                        'text': f"{g['name']}s {direction} shorts on {inst} — proportion moved {short_shift:+.1%} of OI, backed by {abs(short_delta):,.0f} contracts",
                        'severity': 'medium',
                    })

    # Sort: high first, then by signal importance
    type_priority = {
        'flip': 0,
        'dealer_absent': 1,
        'dealer_unusual': 2,
        'divergence': 3,
        'ratio_extreme': 4,
        'crowding': 5,
        'shift': 6,
    }
    return sorted(
        insights,
        key=lambda x: (
            0 if x['severity'] == 'high' else 1,
            type_priority.get(x['type'], 99),
        ),
    )

In [42]:
insights = generate_insights(cot_clean)
insights

[{'type': 'flip',
  'instrument': 'BTC',
  'text': 'Dealers flipped BTC to net short this week',
  'severity': 'high'},
 {'type': 'flip',
  'instrument': 'CAD',
  'text': 'Asset Managers flipped CAD to net short this week',
  'severity': 'high'},
 {'type': 'flip',
  'instrument': 'NASDAQ',
  'text': 'Dealers flipped NASDAQ to net long this week',
  'severity': 'high'},
 {'type': 'dealer_absent',
  'instrument': 'BTC',
  'text': 'Dealers pulling back on BTC — crowding at 8% of OI (8th percentile). Reduced hedging activity',
  'severity': 'high'},
 {'type': 'dealer_absent',
  'instrument': 'CAD',
  'text': 'Dealers pulling back on CAD — crowding at 43% of OI (4th percentile). Reduced hedging activity',
  'severity': 'high'},
 {'type': 'dealer_unusual',
  'instrument': 'US10Y',
  'text': 'Dealers extremely net short on US10Y (0th percentile) — heavy hedging or directional bet',
  'severity': 'high'},
 {'type': 'ratio_extreme',
  'instrument': 'BTC',
  'text': 'Asset Manager long/short rat

In [43]:
insights

[{'type': 'flip',
  'instrument': 'BTC',
  'text': 'Dealers flipped BTC to net short this week',
  'severity': 'high'},
 {'type': 'flip',
  'instrument': 'CAD',
  'text': 'Asset Managers flipped CAD to net short this week',
  'severity': 'high'},
 {'type': 'flip',
  'instrument': 'NASDAQ',
  'text': 'Dealers flipped NASDAQ to net long this week',
  'severity': 'high'},
 {'type': 'dealer_absent',
  'instrument': 'BTC',
  'text': 'Dealers pulling back on BTC — crowding at 8% of OI (8th percentile). Reduced hedging activity',
  'severity': 'high'},
 {'type': 'dealer_absent',
  'instrument': 'CAD',
  'text': 'Dealers pulling back on CAD — crowding at 43% of OI (4th percentile). Reduced hedging activity',
  'severity': 'high'},
 {'type': 'dealer_unusual',
  'instrument': 'US10Y',
  'text': 'Dealers extremely net short on US10Y (0th percentile) — heavy hedging or directional bet',
  'severity': 'high'},
 {'type': 'ratio_extreme',
  'instrument': 'BTC',
  'text': 'Asset Manager long/short rat

In [26]:
variable = 'Levered Net'
tf_in_weeks = 1  #Do this one
treemap_data = cot_treemap(variable, tf_in_weeks, cot_clean, prices)
treemap_data

,index,Levered Net,parent,ticker,return,absolute value
0,1 Month US,-235635.0,,SHV,0.067230,235635.0
1,3 Month US,-960365.0,,SHV,0.067230,960365.0
2,BTC,25985.0,,BTC-USD,5.433403,25985.0
3,CAD,-47911.0,,CAD=X,-0.119900,47911.0
4,CHF,-1225.0,,CHF=X,-0.222538,1225.0
5,GBP,24653.0,,GBPUSD=X,0.465840,24653.0
6,NASDAQ,-140616.0,,QQQ,1.976859,140616.0
7,SP500,-199641.0,,^GSPC,1.352988,199641.0
8,US10Y,-2297078.0,,IEF,0.133510,2297078.0
9,YEN,-60210.0,,JPY=X,-0.098851,60210.0
